In [5]:
import numpy as np
import pandas as pd
from collections import defaultdict
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from utils import *
import tabulate
import csv
import numpy.ma as ma

**Reading the datas**

In [6]:
item_train = pd.read_csv('data/content_item_train.csv',header=None)
user_train = pd.read_csv('data/content_user_train.csv',header=None)
y_train    = pd.read_csv('data/content_y_train.csv',header=None)  #the rating value of the movies rated by the user
item_headers = pd.read_csv('data/content_item_train_header.txt')  #features of the movie
user_headers = pd.read_csv('data/content_user_train_header.txt')  #features of the user
item_vecs = pd.read_csv('data/content_item_vecs.csv',header=None)
movie_list = pd.read_csv('data/content_movie_list.csv')
print(user_headers)
print(item_headers)
user_index = 3  #we start with index 3 for user dataset while training and evaluating the model, skipping the user id, rating count and average rating of user
item_index = 1  #we start with index 1 for item dataset while training and evaluating the model, skipping the movie id


Empty DataFrame
Columns: [user id, rating count, rating ave, Action, Adventure, Animation, Children, Comedy, Crime, Documentary, Drama, Fantasy, Horror, Mystery, Romance, Sci-Fi, Thriller]
Index: []
Empty DataFrame
Columns: [movie id, year, ave rating, Action, Adventure, Animation, Children, Comedy, Crime, Documentary, Drama, Fantasy, Horror, Mystery, Romance, Sci-Fi, Thriller]
Index: []


**Features of users and items**

In [7]:
user_features = list(user_headers.columns)
item_features = list(item_headers.columns)
print(f'The features of item are: {item_features}')
print(f'The features of user are: {user_features}')

The features of item are: ['movie id', 'year', 'ave rating', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Horror', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller']
The features of user are: ['user id', 'rating count', 'rating ave', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Horror', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller']


**making a dictionary of movie id as key and title and genre as the values**

In [8]:
movie_list_array = np.array(movie_list)
movie_dict = defaultdict(dict)
#here based on the movie id , we are storing the title of the movie and the genre
for row in movie_list_array:
    movie_dict[row[0]]['title'] = row[1] 
    movie_dict[row[0]]['genre'] = row[2]             

**Counting number of user and item features**

In [9]:
num_user_features = len(user_features) - 3  #removing userid, rating_count and average rating
num_item_features = len(item_features) - 1  #removing movieid and we didnot remove year , cause year represents the era of a movie
print('The number of user features is:', num_user_features)
print('The number of item features is:', num_item_features)

The number of user features is: 14
The number of item features is: 16


**Converting the item,user and y dataframe into numpy array**

In [10]:
item_train_array = np.array(item_train)
user_train_array = np.array(user_train)
y_train_array = np.array(y_train)


**Splitting datas into train and test dataset**

In [11]:
item_train,item_test = train_test_split(item_train_array,train_size = 0.80, shuffle=True, random_state = 1)  #datas of user
user_train,user_test = train_test_split(user_train_array,train_size = 0.80, shuffle=True, random_state = 1)  #datas of movie
y_train,y_test = train_test_split(y_train_array,train_size = 0.80, shuffle=True, random_state = 1)           #rating data


**Standard scaling of user, item dataset and minmax scaling of rating value**

In [12]:
user_scalar = StandardScaler()
user_scalar.fit(user_train)
user_train_scaled = user_scalar.transform(user_train)  #scaling the user training dataset
user_test_scaled = user_scalar.transform(user_test)    #scaling the user test dataset

item_scalar = StandardScaler()
item_scalar.fit(item_train)
item_train_scaled = item_scalar.transform(item_train)  #scaling the user training dataset
item_test_scaled = item_scalar.transform(item_test)    #scaling the user test dataset

y_scalar = MinMaxScaler((-1,1))
y_scalar.fit(y_train)
y_train_scaled = y_scalar.transform(y_train)  #scaling the user training dataset
y_test_scaled = y_scalar.transform(y_test)    #scaling the user test dataset

**Neural Network Architecture**



In [13]:
num_outputs = 32  #the number of final output of our neural network
tf.random.set_seed(1)
user_nn = tf.keras.models.Sequential([
    tf.keras.layers.Dense(256,activation='relu'),
    tf.keras.layers.Dense(128,activation='relu'),
    tf.keras.layers.Dense(num_outputs)
])
item_nn = tf.keras.models.Sequential([
    tf.keras.layers.Dense(256,activation='relu'),
    tf.keras.layers.Dense(128,activation='relu'),
    tf.keras.layers.Dense(num_outputs)
])
input_user = tf.keras.layers.Input(shape =(num_user_features,))  
vu = user_nn(input_user) #passing the number of user features in the user neural network inorder to get the user vector
vu = tf.keras.ops.normalize(vu, axis=1)  #applying the l2 normalization on user vector

input_item = tf.keras.layers.Input(shape =(num_item_features,))  
vm = item_nn(input_item) #passing the number of item features in the item neural network
vm = tf.keras.ops.normalize(vm, axis=1)  #applying the l2 normalization on item vector

output = tf.keras.layers.Dot(axes=1)([vu,vm])
model = tf.keras.Model([input_user, input_item], output)  #creating the neural network model with the passed input and output

model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 14)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_2 (InputLayer)    │ (None, 16)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ sequential (Sequential)       │ (None, 32)                │          40,864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ sequential_1 (Sequential)     │ (None, 32)                │          41,376 │ input_layer_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ normalize (Normalize)         │ (None, 32)                │               0 │ sequential[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ normalize_1 (Normalize)       │ (None, 32)                │               0 │ sequential_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dot (Dot)                     │ (None, 1)                 │               0 │ normalize[0][0],           │
│                               │                           │                 │ normalize_1[0][0]          │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 82,240 (321.25 KB)

 Trainable params: 82,240 (321.25 KB)

 Non-trainable params: 0 (0.00 B)

**Training the neural network**

In [14]:
tf.random.set_seed(1)
cost_function = tf.keras.losses.MeanSquaredError()  #we are using MSE as a cost function for our nn model
opt = keras.optimizers.Adam(learning_rate = 0.01)
model.compile(optimizer = opt, loss = cost_function)  #setting the cost function as well as the optimization for the neural network

model.fit([user_train_scaled[:,user_index:], item_train_scaled[:,item_index:]],y_train_scaled,epochs = 30)  #training the neural network model, and we are only selecting the required features from both user and item training dataset
model.evaluate([user_test_scaled[:,user_index:], item_test_scaled[:,item_index:]],y_test_scaled)

Epoch 1/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - loss: 0.1236
Epoch 2/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 15s 11ms/step - loss: 0.1143
Epoch 3/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.1087
Epoch 4/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - loss: 0.1049
Epoch 5/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - loss: 0.1020
Epoch 6/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - loss: 0.0999
Epoch 7/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - loss: 0.0979
Epoch 8/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 20s 10ms/step - loss: 0.0958
Epoch 9/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - loss: 0.0938
Epoch 10/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - loss: 0.0921
Epoch 11/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - loss: 0.0903
Epoch 12/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 17s 14ms/step - loss: 0.0887
Epoch 13/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - loss: 0.0872
Epoch 14/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - loss: 0.0857
Ep

0.08311344683170319

**Creation of new user ratings**

In [15]:
new_user_id = 5000
new_rating_ave = 0.0
new_action = 0.0
new_adventure = 5.0
new_animation = 0.0
new_childrens = 0.0
new_comedy = 0.0
new_crime = 0.0
new_documentary = 0.0
new_drama = 0.0
new_fantasy = 5.0
new_horror = 0.0
new_mystery = 0.0
new_romance = 0.0
new_scifi = 0.0
new_thriller = 0.0
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])

In [16]:
item_vecs_array = np.array(item_vecs)  #this is the item dataset which we will use for the predition of ratings done by the new user
item_vecs_array

array([[4.05400000e+03, 2.00100000e+03, 2.84375000e+00, ...,
        1.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [4.06900000e+03, 2.00100000e+03, 2.90909091e+00, ...,
        1.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [4.14800000e+03, 2.00100000e+03, 2.93589744e+00, ...,
        0.00000000e+00, 0.00000000e+00, 1.00000000e+00],
       ...,
       [1.77765000e+05, 2.01700000e+03, 3.53846154e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.79819000e+05, 2.01700000e+03, 3.12500000e+00, ...,
        0.00000000e+00, 1.00000000e+00, 0.00000000e+00],
       [1.87593000e+05, 2.01800000e+03, 3.87500000e+00, ...,
        0.00000000e+00, 1.00000000e+00, 0.00000000e+00]], shape=(847, 17))

In [17]:
user_vecs = gen_newuser_data(user_vec,len(item_vecs_array))  #here we are making the same size of user vector to match with that of item vector or movie dataset
user_vecs

array([[5.e+03, 3.e+00, 0.e+00, ..., 0.e+00, 0.e+00, 0.e+00],
       [5.e+03, 3.e+00, 0.e+00, ..., 0.e+00, 0.e+00, 0.e+00],
       [5.e+03, 3.e+00, 0.e+00, ..., 0.e+00, 0.e+00, 0.e+00],
       ...,
       [5.e+03, 3.e+00, 0.e+00, ..., 0.e+00, 0.e+00, 0.e+00],
       [5.e+03, 3.e+00, 0.e+00, ..., 0.e+00, 0.e+00, 0.e+00],
       [5.e+03, 3.e+00, 0.e+00, ..., 0.e+00, 0.e+00, 0.e+00]],
      shape=(847, 17))

**Scaling of user as well as movie vector**

In [18]:
#using the trained scalar to transform both the user as well as item vector
scaled_user_vecs = user_scalar.transform(user_vecs)  
scaled_user_vecs
scaled_item_vecs = item_scalar.transform(item_vecs_array)
scaled_item_vecs

array([[-0.94788869, -1.1989544 , -1.98286105, ...,  2.49684357,
        -0.50803289, -0.6593247 ],
       [-0.94749159, -1.1989544 , -1.81467973, ...,  2.49684357,
        -0.50803289, -0.6593247 ],
       [-0.94540021, -1.1989544 , -1.74568226, ..., -0.40050567,
        -0.50803289,  1.51670339],
       ...,
       [ 3.65079032,  2.82057147, -0.19473926, ..., -0.40050567,
        -0.50803289, -0.6593247 ],
       [ 3.70516619,  2.82057147, -1.25895015, ..., -0.40050567,
         1.9683765 , -0.6593247 ],
       [ 3.91096853,  3.07179184,  0.67147891, ..., -0.40050567,
         1.9683765 , -0.6593247 ]], shape=(847, 17))

**Making prediction or recommendation of movies to this new user**

In [19]:
Y_value = model.predict([scaled_user_vecs[:,user_index:], scaled_item_vecs[:,item_index:]])  #prediction of model takes only one argument
#since while training the model, the rating value was scaled.
#we are unscaling the rating value here 
unscaled_y_value = y_scalar.inverse_transform(Y_value)
print('The five predicted rating is:', unscaled_y_value[:5])


27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
The five predicted rating is: [[2.3157032]
 [2.4369624]
 [2.0822246]
 [2.333638 ]
 [2.2036092]]


**Sorting the predicted rating values**

In [20]:
sorted_indexes = np.argsort(-unscaled_y_value,axis = 0).reshape(-1).tolist() #here we are saving the indexes of the rating value based on descending order, and we are using -ve sign cause .argsort sort values in ascending order 
sorted_rating_values = unscaled_y_value[sorted_indexes]  #predicted rating values by the new user on the movies, sorted from highest to lowest


**Displaying the predicted top 10 movies rated by the user**

In [21]:
sorted_movies = item_vecs_array[sorted_indexes]  #here we are sorting the movies based on the predicted rating values from highest to lowest
display = [["predicted_rating","movie id", "year","average_rating", "title", "genres"]]
for i in range(10):
    predicted_rating = sorted_rating_values[i]
    movie_id = sorted_movies[i,0].astype(int)  #extracting the movie_id
    year = sorted_movies[i,1]
    average_rating = sorted_movies[i,2].astype(int)  
    movie_title = movie_dict[movie_id]['title']
    movie_genre = movie_dict[movie_id]['genre']
    display.append([predicted_rating,movie_id,year,average_rating,movie_title,movie_genre])
table = tabulate.tabulate(display, tablefmt='html', headers="firstrow")
table

    
    
    
    

predicted_rating,movie id,year,average_rating,title,genres
4.20216,137857,2016,3,The Jungle Book (2016),Adventure|Drama|Fantasy
4.18412,98809,2012,3,"Hobbit: An Unexpected Journey, The (2012)",Adventure|Fantasy
4.12573,4896,2001,3,Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001),Adventure|Children|Fantasy
4.11591,106489,2013,3,"Hobbit: The Desolation of Smaug, The (2013)",Adventure|Fantasy
4.08191,5816,2002,3,Harry Potter and the Chamber of Secrets (2002),Adventure|Fantasy
4.07195,54001,2007,3,Harry Potter and the Order of the Phoenix (2007),Adventure|Drama|Fantasy
4.05369,8368,2004,3,Harry Potter and the Prisoner of Azkaban (2004),Adventure|Fantasy
4.04575,41566,2005,3,"Chronicles of Narnia: The Lion, the Witch and the Wardrobe, The (2005)",Adventure|Children|Fantasy
4.04258,69844,2009,3,Harry Potter and the Half-Blood Prince (2009),Adventure|Fantasy|Mystery|Romance
4.02509,59387,2006,3,"Fall, The (2006)",Adventure|Drama|Fantasy


**Creating a separate item model using the trained 2 tower neural network model**

In [22]:
input_item = tf.keras.layers.Input(shape =(num_item_features,))   #making the input layer for neural network model 
vm = item_nn(input_item) #passing the number of item features in the item neural network
vm = tf.keras.ops.normalize(vm, axis=1)  #applying the L2 normalization feature wise
model_m = tf.keras.Model( input_item, vm)  #creating the model with the passed input and output
model_m.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)           │ (None, 16)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ sequential_1 (Sequential)            │ (None, 32)                  │          41,376 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ normalize_2 (Normalize)              │ (None, 32)                  │               0 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 41,376 (161.62 KB)

 Trainable params: 41,376 (161.62 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
scaled_item_vecs = item_scalar.transform(item_vecs_array)
vms=model_m.predict([scaled_item_vecs[:,item_index:]])  #predicting the movie vectors


18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step  

C:\pythonprojects\Machine learning\unsupervised_learning\content_based_recommendation\venv\lib\site-packages\keras\src\models\functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor_15
Received: inputs=('Tensor(shape=(32, 16))',)
  warnings.warn(msg)
C:\pythonprojects\Machine learning\unsupervised_learning\content_based_recommendation\venv\lib\site-packages\keras\src\models\functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor_15
Received: inputs=('Tensor(shape=(None, 16))',)
  warnings.warn(msg)


27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step


In [24]:
vms

array([[ 0.10641509,  0.27998477,  0.25875875, ...,  0.02368528,
        -0.16119395, -0.02063369],
       [ 0.02996543,  0.10810671,  0.20204276, ...,  0.06583225,
        -0.22526918, -0.0268344 ],
       [-0.12746337,  0.21895558,  0.05766075, ..., -0.00538183,
         0.00793561, -0.12767144],
       ...,
       [ 0.01541685, -0.12393122, -0.08902837, ..., -0.22473942,
         0.17860526,  0.2048513 ],
       [ 0.13625489, -0.08743322,  0.14047788, ...,  0.02247081,
        -0.19101405,  0.09173565],
       [-0.05181096, -0.2577547 , -0.02874717, ...,  0.04764351,
        -0.04791426,  0.15164626]], shape=(847, 32), dtype=float32)

**Square distance between two vectors, checking the similarity between two vectors**

In [25]:
def square_distance(a,b):
    return np.sum(np.square(a-b))  #returning the square distance between the two item vectors
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {square_distance(a1, b1):0.3f}")
print(f"squared distance between a2 and b2: {square_distance(a2, b2):0.3f}")
print(f"squared distance between a3 and b3: {square_distance(a3, b3):0.3f}")        

squared distance between a1 and b1: 0.000
squared distance between a2 and b2: 0.030
squared distance between a3 and b3: 2.000


**Finding the similar movies for 50 movies from the movies vector**

In [26]:
count = 50  # number of movies to display
dim = len(vms)  #dimension of the distance storage matrix
dist = np.zeros((dim,dim))  #matrix that stores the distance between two movies

#this loop is used for storing the distance between every movie pairs possible in the dist matrix of shape dim * dim
for i in range(dim):
    for j in range(dim):
        dist[i,j] = square_distance(vms[i, :], vms[j, :])  #storing the distance between movie i and j 
        
#here we are masking the diagonal in the distance storage matrix inorder to prevent from getting the same movie while finding the most similar movie based on the square distance between each other
masked_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # masking the diagonal or ignoring the diagonal using the identity matrix of size of dist's rows

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(masked_dist[i])  #here we are finding the index of the movie which has the most minimum square distance from the current ith movie
    movie1_id = int(item_vecs_array[i,0])   #finding the id of current index movie
    movie2_id = int(item_vecs_array[min_idx,0])  #then using that index , we can easily find the id of the most similar movie or movie having the least square distance from the current i indexed movie
    disp.append( [movie_dict[movie1_id]['title'], movie_dict[movie1_id]['genre'],
                  movie_dict[movie2_id]['title'], movie_dict[movie1_id]['genre']]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow")
table

movie1,genres,movie2,genres
Save the Last Dance (2001),Drama|Romance,Mona Lisa Smile (2003),Drama|Romance
"Wedding Planner, The (2001)",Comedy|Romance,Mr. Deeds (2002),Comedy|Romance
Hannibal (2001),Horror|Thriller,Final Destination 2 (2003),Horror|Thriller
Saving Silverman (Evil Woman) (2001),Comedy|Romance,Down with Love (2003),Comedy|Romance
Down to Earth (2001),Comedy|Fantasy|Romance,Bewitched (2005),Comedy|Fantasy|Romance
"Mexican, The (2001)",Action|Comedy,Rush Hour 2 (2001),Action|Comedy
15 Minutes (2001),Thriller,Panic Room (2002),Thriller
Enemy at the Gates (2001),Drama,Seabiscuit (2003),Drama
Heartbreakers (2001),Comedy|Crime|Romance,Fun with Dick and Jane (2005),Comedy|Crime|Romance
Spy Kids (2001),Action|Adventure|Children|Comedy,Scooby-Doo (2002),Action|Adventure|Children|Comedy
